In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.sparse import csc_matrix, eye, diags
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.preprocessing import MinMaxScaler

In [ ]:
def WhittakerSmooth(x, w, lambda_, differences=1):
    X = np.matrix(x)
    m = X.size
    E = eye(m, format='csc')
    for i in range(differences):
        E = E[1:] - E[:-1]  
    W = diags(w, 0, shape=(m, m))
    A = csc_matrix(W + (lambda_ * E.T * E))
    B = csc_matrix(W * X.T)
    background = spsolve(A, B)
    return np.array(background)
    
def airPLS(x, lambda_=4, porder=1, itermax=15):
    m = x.shape[0]
    w = np.ones(m)
    for i in range(1, itermax + 1):
        z = WhittakerSmooth(x, w, lambda_, porder)
        d = x - z
        dssn = np.abs(d[d < 0].sum())
        if (dssn < 0.001 * (abs(x)).sum() or i == itermax):
            if (i == itermax): print('WARING max iteration reached!')
            break
        w[d >= 0] = 0  
        w[d < 0] = np.exp(i * np.abs(d[d < 0]) / dssn)
        w[0] = np.exp(i * (d[d < 0]).max() / dssn)
        w[-1] = w[0]
    return z
    
def AirPLS(data):
    for i in range(data.shape[0]):
        background_removed = data.iloc[i] - airPLS(data.iloc[i])
        background_removed[background_removed < 0] = 0
        data.iloc[i] = background_removed
    return data

def to_normalize(data_raw, rs_raw, areas):

    data_raw_temp = pd.DataFrame(data_raw).T
    rs_raw_temp = pd.DataFrame(rs_raw)
    print(data_raw_temp.shape)
    
    if len(data_raw_temp.columns) == 0:
        return data_raw
    if len(rs_raw_temp.columns) == 0:
        return data_raw
    
    scaler = MinMaxScaler(feature_range=(0, 1))  
    data_new = data_raw_temp.copy()
    min_ind = 0
    max_ind = 0
    for area in areas:
        min_rs = area[0]
        max_rs = area[1]
        print(min_rs)
        print(max_rs)

        for i in range(rs_raw_temp.shape[1]):
            if rs_raw_temp.iloc[0,i] > min_rs:
                min_ind = i
                break
        for i in range(rs_raw_temp.shape[1]):
            if rs_raw_temp.iloc[0,i] == max_rs:
                max_ind = i
                break
            elif rs_raw_temp.iloc[0,i] > max_rs:
                max_ind = i-1
                break
            else:
                max_ind = i
        data_new.iloc[min_ind : max_ind+1, :] = scaler.fit_transform(np.array(data_raw_temp.iloc[min_ind : max_ind+1, :]))#归一化
        print(np.array(data_raw_temp.iloc[min_ind : max_ind+1, :]))
    
    return data_new.T

In [ ]:
file_path = 'path/raw.xlsx'
data = pd.read_excel(file_path, sheet_name='Sheet1')
id_label_data = data[['ID', 'Label']] 
data = data.iloc[:, 2:]
original_columns = data.columns

data_AirPlS = AirPLS(data.copy())
data_SG = signal.savgol_filter(data_AirPlS.values, 21, 5)
data_SG[data_SG < 0] = 0
smooth_data_df = pd.DataFrame(
    data_SG,
    index=data.index,
    columns=original_columns
)
cols = smooth_data_df.columns.astype(float)
mask = ~((cols >= 1800) & (cols <= 2800))
data_raw = smooth_data_df.loc[:, mask]
rs_raw = pd.DataFrame([data_raw.columns.values])
areas = [[500, 3500]]
normalized_data = to_normalize(data_raw, rs_raw, areas)
final_data = pd.concat([id_label_data, normalized_data], axis=1)

save_dir = 'path/'
os.makedirs(save_dir, exist_ok=True)
final_data.to_excel(os.path.join(save_dir, 'NOR.xlsx'), index=False)